Praktikum Sederhana LTSM Week 12 Sesi 1 

Nama    : Gresia Delia Silalahi

NIM     : 42324011

Topik   : LSTM pada dataset 'RSCASN.csv'

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.dates as mdates

In [5]:
# CODE PRAKTIKUM SEDERHANA (TANPA OUTPUT)

# 1. Membaca dataset penjualan ritel & membagi data latih (316 bulan) dan uji (18 bulan)
sales_data = pd.read_csv('RSCCASN.csv', parse_dates=True, index_col='DATE')
sales_data.columns = ['Sales']
train = sales_data.iloc[:-18]
test = sales_data.iloc[-18:].copy()

# 2. Normalisasi data penjualan ke skala 0 sampai 1
scaler = MinMaxScaler()
scaler.fit(train)
scaled_train = scaler.transform(train)

# 3. Membuat jendela waktu (12 bulan input -> prediksi bulan ke-13)
length = 12
n_features = 1
generator = TimeseriesGenerator(
    scaled_train,
    scaled_train,
    length=length,
    batch_size=1
)

# 4. Membangun arsitektur model LSTM + Dropout + Dense(1)
model_lstm = Sequential()
model_lstm.add(Input(shape=(length, n_features)))
model_lstm.add(LSTM(100, activation='relu'))
model_lstm.add(Dropout(0.2))
model_lstm.add(Dense(1))

model_lstm.compile(optimizer='adam', loss='mse')
model_lstm.fit(generator, epochs=10)

prediksi_scaled = []
current_batch = scaled_train[-length:].reshape((1, length, n_features))

for i in range(len(test)):
    pred_step = model_lstm.predict(current_batch, verbose=0)[0]
    prediksi_scaled.append(pred_step)
    current_batch = np.append(current_batch[:, 1:, :], [[pred_step]], axis=1)

# 5. Mengembalikan hasil prediksi (skala 0-1) ke angka penjualan asli
hasil_prediksi = scaler.inverse_transform(prediksi_scaled)

Epoch 1/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.0255
Epoch 2/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0209
Epoch 3/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0161
Epoch 4/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0100
Epoch 5/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0060
Epoch 6/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0047
Epoch 7/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0034
Epoch 8/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0023
Epoch 9/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0038
Epoch 10/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.0028


In [6]:
import numpy as np
import pandas as pd

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# Input dari normalisasi MinMaxScaler pada data RSCCASN.csv
X_t, X_min, X_max = 28973, 6938, 34482
x_t = round((X_t - X_min) / (X_max - X_min), 4)  # 0.8000
h_prev = 0.2
C_prev = 0.6

# Bobot dan bias sederhana untuk 1 sel LSTM + lapisan Dense(1)
W_fh, W_fx, b_f = 0.5, 0.5, 0.5
W_ih, W_ix, b_i = 0.5, 0.5, 0.3
W_ch, W_cx, b_c = 0.5, 0.5, 0.0
W_oh, W_ox, b_o = 0.5, 0.5, 0.1
W_y, b_y = 1.0, 0.3

# Perhitungan Python (eksak)
f_t = sigmoid(W_fh * h_prev + W_fx * x_t + b_f)
i_t = sigmoid(W_ih * h_prev + W_ix * x_t + b_i)
c_tilde = np.tanh(W_ch * h_prev + W_cx * x_t + b_c)
C_t = f_t * C_prev + i_t * c_tilde
o_t = sigmoid(W_oh * h_prev + W_ox * x_t + b_o)
h_t = o_t * np.tanh(C_t)
y_hat_scaled = W_y * h_t + b_y

# Bandingkan dengan hasil perhitungan manual di kertas (4 desimal)
manual_aproksimasi = [0.7311, 0.6900, 0.4621, 0.7575, 0.6457, 0.4130, 0.7130]
hasil_python = [
    round(float(val), 4)
    for val in [f_t, i_t, c_tilde, C_t, o_t, h_t, y_hat_scaled]
]

df_perbandingan = pd.DataFrame({
    'Proses': [
        'Forget Gate',
        'Input Gate',
        'Candidate Cell State',
        'Cell State',
        'Output Gate',
        'Hidden State',
        'Prediksi Dense(1)'
    ],
    'Manual (aproksimasi)': manual_aproksimasi,
    'Hasil Python': hasil_python,
})
df_perbandingan['Cocok'] = (
    df_perbandingan['Manual (aproksimasi)'] == df_perbandingan['Hasil Python']
)

print(df_perbandingan)
print("Semua hasil cocok:", df_perbandingan['Cocok'].all())

                 Proses  Manual (aproksimasi)  Hasil Python  Cocok
0           Forget Gate                0.7311        0.7311   True
1            Input Gate                0.6900        0.6900   True
2  Candidate Cell State                0.4621        0.4621   True
3            Cell State                0.7575        0.7575   True
4           Output Gate                0.6457        0.6457   True
5          Hidden State                0.4130        0.4130   True
6     Prediksi Dense(1)                0.7130        0.7130   True
Semua hasil cocok: True
